# Offline batch inference with vLLM: the boring 10x

Companion notebook for the GDE #9 post. All numbers are a toy queueing model calibrated to cited published measurements (see post) — no GPU is needed to run this notebook. The real vLLM calls are in the last cell and are skipped gracefully when vLLM is not installed.

In [ ]:
"""Cell 1: throughput vs batch size.
Calibration: A100 80GB, Llama-3.1-8B, 256 output tokens.
HF sequential ~50-80 tok/s; vLLM offline batch ~300-500 tok/s (smazcw3.github.io, 2025).
Model: vLLM throughput saturates log-like as the KV cache / scheduler fills."""
import numpy as np
bs = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256])
vllm_tps = np.minimum(420 * (1 - np.exp(-bs / 24.0)) + 8, 420)
hf_tps = np.full_like(bs, 65.0, dtype=float)
for b, v, h in zip(bs, vllm_tps, hf_tps):
    print(f'batch {b:>3}: vLLM {v:6.1f} tok/s   HF {h:5.1f} tok/s   speedup {v/h:.2f}x')
assert 300 <= vllm_tps[-1] <= 500, 'saturation must land in the cited 300-500 band'
assert vllm_tps[6] / hf_tps[6] > 4.0, 'batch 64 must beat sequential by >4x'
assert vllm_tps[-1] / vllm_tps[0] > 5, 'batching must buy >5x over batch-1 vLLM'
print('OK: vLLM saturates in the cited band; batch 64 is %.2fx HF sequential' % (vllm_tps[6]/hf_tps[6]))

In [ ]:
"""Cell 2: the TTFT trap.
Saumilya (133-benchmark sweep, V100 4-bit, 62 prompts): HF TTFT 594-736 ms,
vLLM offline batch TTFT 27,775 ms (Qwen) / 47,297 ms (Llama).
vLLM online server (vllm bench serve, Hermes-3-8B): mean TTFT 71.5 ms.
Offline batch TTFT is dominated by waiting for the whole batch to assemble."""
hf_ttft_s = 0.65        # mean of 594/736 ms
vllm_offline_ttft_s = 47.3   # Llama-3.1-8B, 62-prompt batch
vllm_online_ttft_s = 0.0715 # vllm bench serve mean
print(f'HF sequential TTFT:      {hf_ttft_s:.2f} s')
print(f'vLLM offline batch TTFT: {vllm_offline_ttft_s:.1f} s  ({vllm_offline_ttft_s/hf_ttft_s:.0f}x HF)')
print(f'vLLM online server TTFT: {vllm_online_ttft_s*1000:.1f} ms  ({hf_ttft_s/vllm_online_ttft_s:.1f}x FASTER than HF)')
assert vllm_offline_ttft_s / hf_ttft_s > 50, 'offline batch TTFT must be >> sequential'
assert vllm_online_ttft_s < hf_ttft_s, 'online server must beat HF on TTFT'
print('OK: same engine, opposite TTFT verdicts - the API (batch vs streaming) is the variable, not the engine')

In [ ]:
"""Cell 3: stacking the knobs (multiplicative, anchored to cited endpoints).
V1 engine: up to 1.7x over V0 (vLLM V1 blog post). Batch packing: 4-6x over sequential.
Prefix cache + request sorting: 1.1-1.44x over stock vLLM (BlendServe/BatchLLM, Parasail).
gpu_memory_utilization 0.90 -> 0.95: ~1.1x more KV headroom. Disabling speculative
decoding above batch ~tens: ~1.15x (draft overhead exceeds gain at high batch)."""
knobs = {'V1 engine': 1.7, 'batch packing': 5.0, 'prefix cache + sorting': 1.3,
         'gpu_mem 0.95': 1.1, 'disable spec decoding at batch': 1.15}
cum = 1.0
for k, m in knobs.items():
    cum *= m
    print(f'{k:32s} x{m:.2f}   cumulative {cum:6.2f}x')
assert abs(cum - 14.0) < 0.6, 'stacked headroom should be ~14x'
realistic = 6.46 * 1.44  # measured saturation speedup (cell 1) x top of BlendServe/BatchLLM sorting range
print(f'Realistic stacked gain over naive sequential: ~{realistic:.1f}x on prefix-heavy workloads (the boring 10x, rounded)')
assert 9 < realistic < 10

In [ ]:
"""Cell 4: cost per 1M output tokens at $2/hr A100 (illustrative list-ish rate)."""
price_per_hr = 2.00
methods = {'HF sequential': 65, 'vLLM online': 383, 'vLLM offline': 420, 'vLLM offline + prefix cache': 546}
for name, tps in methods.items():
    cost = price_per_hr / tps * 1e6 / 3600
    print(f'{name:32s} {tps:>4} tok/s   ${cost:.2f} / 1M tokens')
seq = price_per_hr / 65 * 1e6 / 3600
off = price_per_hr / 546 * 1e6 / 3600
assert off < 1.10, 'offline+cache must be ~$1/1M tokens at this rate'
assert seq / off > 6, 'offline+cache must be >6x cheaper than sequential'
print(f'OK: offline batch + prefix cache is {seq/off:.1f}x cheaper per token than sequential HF')

In [ ]:
"""Cell 5: the real vLLM code (runs only where vLLM + a GPU are installed).
On this machine it prints the setup commands instead of failing."""
try:
    from vllm import LLM, SamplingParams
    print('vLLM installed - run this on a GPU box:')
    print('  llm = LLM(model="meta-llama/Llama-3.1-8B-Instruct", gpu_memory_utilization=0.95,')
    print('            max_num_batched_tokens=16384, enable_prefix_caching=True)')
    print('  sp = SamplingParams(temperature=0.0, max_tokens=256)')
    print('  outs = llm.generate(prompts, sp)   # prompts sorted by shared prefix first')
except ImportError:
    print('vLLM not installed here (expected - this is a CPU box). Install + run on GPU:')
    print('  pip install vllm   # 0.33+ for the V1 engine defaults used in the post')
    print('  python -c "from vllm import LLM, SamplingParams; print(\'ok\')"')